# LeNet-5 pour la reconnaissance des caractères Tifinagh (33 classes)
Implémentation **NumPy pur** (sans PyTorch/TensorFlow) : couches, rétropropagation, optimiseurs (SGD, Momentum, Nesterov, RMSprop, Adam), entraînement, évaluation et visualisations.

**Mode d'emploi** : exécuter les cellules dans l'ordre (*Exécution > Tout exécuter*). Dans la cellule *Données*, choisir la source :
- `SOURCE = "zip"` : fichier `.zip` de la base AMHCD (sur Colab : le téléverser via le panneau de fichiers à gauche) ;
- `SOURCE = "dossier"` : dossier déjà décompressé (un sous-dossier par classe) ;
- `SOURCE = "synthetique"` : jeu artificiel pour tester le notebook sans la base (résultats **sans valeur scientifique**).

In [ ]:
import os, json, time
import numpy as np
import matplotlib.pyplot as plt
import csv
from numpy.lib.stride_tricks import sliding_window_view
from IPython.display import Image, display
print('NumPy', np.__version__)

## 1. Couches (forward + backward)
Convolution (im2col), pooling moyen/max, fully connected, dropout, activations, softmax + entropie croisée.

In [ ]:


# --------------------------------------------------------------------------
# Initialisation des poids
# --------------------------------------------------------------------------
def init_weights(shape, fan_in, fan_out, scheme="lecun", rng=None):
    rng = rng or np.random.default_rng(0)
    if scheme == "xavier":
        lim = np.sqrt(6.0 / (fan_in + fan_out))
        return rng.uniform(-lim, lim, shape)
    if scheme == "he":
        return rng.normal(0.0, np.sqrt(2.0 / fan_in), shape)
    if scheme == "lecun":
        return rng.normal(0.0, np.sqrt(1.0 / fan_in), shape)
    raise ValueError(f"Initialisation inconnue : {scheme}")


class Layer:
    trainable = False

    def __init__(self, name=None):
        self.name = name or self.__class__.__name__
        self.params, self.grads = {}, {}
        self.training = True

    def forward(self, x):
        raise NotImplementedError

    def backward(self, dout):
        raise NotImplementedError


# --------------------------------------------------------------------------
# Convolution
# --------------------------------------------------------------------------
class Conv2D(Layer):
    """Convolution 2D (im2col) : out = W * x + b, avec stride et padding."""
    trainable = True

    def __init__(self, in_ch, out_ch, k, stride=1, padding=0, init="lecun",
                 rng=None, dtype=np.float32, name=None):
        super().__init__(name)
        self.in_ch, self.out_ch, self.k = in_ch, out_ch, k
        self.stride, self.padding = stride, padding
        W = init_weights((out_ch, in_ch, k, k), in_ch * k * k, out_ch * k * k, init, rng)
        self.params["W"] = W.astype(dtype)
        self.params["b"] = np.zeros(out_ch, dtype=dtype)

    def forward(self, x):
        p, k, s, F = self.padding, self.k, self.stride, self.out_ch
        if p:
            x = np.pad(x, ((0, 0), (0, 0), (p, p), (p, p)))
        N, C, H, W = x.shape
        win = sliding_window_view(x, (k, k), axis=(2, 3))[:, :, ::s, ::s]
        oh, ow = win.shape[2], win.shape[3]
        cols = win.transpose(0, 2, 3, 1, 4, 5).reshape(N * oh * ow, C * k * k)
        Wm = self.params["W"].reshape(F, -1)
        out = cols @ Wm.T + self.params["b"]
        out = out.reshape(N, oh, ow, F).transpose(0, 3, 1, 2)
        self._cache = (cols, x.shape, oh, ow)
        return np.ascontiguousarray(out)

    def backward(self, dout):
        cols, xshape, oh, ow = self._cache
        N, C, H, W = xshape
        k, s, p, F = self.k, self.stride, self.padding, self.out_ch
        d2 = dout.transpose(0, 2, 3, 1).reshape(-1, F)
        Wm = self.params["W"].reshape(F, -1)
        self.grads["W"] = (d2.T @ cols).reshape(self.params["W"].shape)
        self.grads["b"] = d2.sum(axis=0)
        dcols = (d2 @ Wm).reshape(N, oh, ow, C, k, k)
        dx = np.zeros(xshape, dtype=dout.dtype)
        for i in range(k):
            for j in range(k):
                dx[:, :, i:i + oh * s:s, j:j + ow * s:s] += \
                    dcols[:, :, :, :, i, j].transpose(0, 3, 1, 2)
        return dx[:, :, p:H - p, p:W - p] if p else dx


# --------------------------------------------------------------------------
# Pooling
# --------------------------------------------------------------------------
class AvgPool2D(Layer):
    """Sous-échantillonnage par moyenne (S2, S4)."""

    def __init__(self, size=2, stride=None, name=None):
        super().__init__(name)
        self.size, self.stride = size, stride or size

    def forward(self, x):
        p, s = self.size, self.stride
        win = sliding_window_view(x, (p, p), axis=(2, 3))[:, :, ::s, ::s]
        self._xshape, self._o = x.shape, win.shape[2:4]
        return win.mean(axis=(4, 5))

    def backward(self, dout):
        p, s = self.size, self.stride
        oh, ow = self._o
        dx = np.zeros(self._xshape, dtype=dout.dtype)
        g = dout / (p * p)
        for i in range(p):
            for j in range(p):
                dx[:, :, i:i + oh * s:s, j:j + ow * s:s] += g
        return dx


class MaxPool2D(Layer):
    """Sous-échantillonnage par maximum (variante)."""

    def __init__(self, size=2, stride=None, name=None):
        super().__init__(name)
        self.size, self.stride = size, stride or size

    def forward(self, x):
        p, s = self.size, self.stride
        win = sliding_window_view(x, (p, p), axis=(2, 3))[:, :, ::s, ::s]
        N, C, oh, ow = win.shape[:4]
        flat = win.reshape(N, C, oh, ow, p * p)
        self._idx = flat.argmax(axis=-1)
        self._xshape, self._o = x.shape, (oh, ow)
        return flat.max(axis=-1)

    def backward(self, dout):
        p, s = self.size, self.stride
        oh, ow = self._o
        dx = np.zeros(self._xshape, dtype=dout.dtype)
        for i in range(p):
            for j in range(p):
                mask = (self._idx == i * p + j)
                dx[:, :, i:i + oh * s:s, j:j + ow * s:s] += dout * mask
        return dx


# --------------------------------------------------------------------------
# Fully connected / utilitaires
# --------------------------------------------------------------------------
class Flatten(Layer):
    def forward(self, x):
        self._shape = x.shape
        return x.reshape(x.shape[0], -1)

    def backward(self, dout):
        return dout.reshape(self._shape)


class Dense(Layer):
    """Couche fully connected : out = x W + b."""
    trainable = True

    def __init__(self, n_in, n_out, init="lecun", rng=None, dtype=np.float32, name=None):
        super().__init__(name)
        self.params["W"] = init_weights((n_in, n_out), n_in, n_out, init, rng).astype(dtype)
        self.params["b"] = np.zeros(n_out, dtype=dtype)

    def forward(self, x):
        self._x = x
        return x @ self.params["W"] + self.params["b"]

    def backward(self, dout):
        self.grads["W"] = self._x.T @ dout
        self.grads["b"] = dout.sum(axis=0)
        return dout @ self.params["W"].T


class Dropout(Layer):
    """Dropout inversé (amélioration optionnelle)."""

    def __init__(self, rate=0.5, rng=None, name=None):
        super().__init__(name)
        self.rate, self.rng = rate, rng or np.random.default_rng(0)

    def forward(self, x):
        if not self.training or self.rate == 0:
            self._mask = None
            return x
        self._mask = (self.rng.random(x.shape) >= self.rate).astype(x.dtype) / (1 - self.rate)
        return x * self._mask

    def backward(self, dout):
        return dout if self._mask is None else dout * self._mask


# --------------------------------------------------------------------------
# Activations
# --------------------------------------------------------------------------
class Tanh(Layer):
    def forward(self, x):
        self._y = np.tanh(x)
        return self._y

    def backward(self, dout):
        return dout * (1.0 - self._y ** 2)


class ReLU(Layer):
    def forward(self, x):
        self._m = x > 0
        return x * self._m

    def backward(self, dout):
        return dout * self._m


class Sigmoid(Layer):
    def forward(self, x):
        self._y = 1.0 / (1.0 + np.exp(-x))
        return self._y

    def backward(self, dout):
        return dout * self._y * (1.0 - self._y)


def get_activation(name, layer_name=None):
    table = {"tanh": Tanh, "relu": ReLU, "sigmoid": Sigmoid}
    if name not in table:
        raise ValueError(f"Activation inconnue : {name}")
    return table[name](name=layer_name)


# --------------------------------------------------------------------------
# Perte : softmax + entropie croisée
# --------------------------------------------------------------------------
class SoftmaxCrossEntropy:
    def forward(self, logits, y):
        z = logits - logits.max(axis=1, keepdims=True)
        e = np.exp(z)
        self.probs = e / e.sum(axis=1, keepdims=True)
        self.y = y
        n = len(y)
        return float(-np.log(self.probs[np.arange(n), y] + 1e-12).mean())

    def backward(self):
        n = len(self.y)
        d = self.probs.copy()
        d[np.arange(n), self.y] -= 1.0
        return d / n


## 2. Optimiseurs
SGD, Momentum, Nesterov, RMSprop, Adam (avec régularisation L2 optionnelle).

In [ ]:


class Optimizer:
    def __init__(self, lr=1e-3, weight_decay=0.0):
        self.lr, self.weight_decay = lr, weight_decay
        self.state, self.t = {}, 0

    def step(self, layers):
        self.t += 1
        for li, layer in enumerate(layers):
            if not layer.trainable:
                continue
            for k, p in layer.params.items():
                g = layer.grads[k]
                if self.weight_decay and k == "W":      # régularisation L2
                    g = g + self.weight_decay * p
                self._update((li, k), p, g)

    def _update(self, key, p, g):
        raise NotImplementedError


class SGD(Optimizer):
    """theta <- theta - lr * g ; avec momentum : v <- mu v - lr g ; theta <- theta + v."""

    def __init__(self, lr=0.01, momentum=0.0, nesterov=False, weight_decay=0.0):
        super().__init__(lr, weight_decay)
        self.mu, self.nesterov = momentum, nesterov

    def _update(self, key, p, g):
        if self.mu == 0:
            p -= self.lr * g
            return
        v = self.state.get(key)
        if v is None:
            v = np.zeros_like(p)
        v_prev = v
        v = self.mu * v - self.lr * g
        self.state[key] = v
        p += (-self.mu * v_prev + (1 + self.mu) * v) if self.nesterov else v


class RMSprop(Optimizer):
    def __init__(self, lr=1e-3, rho=0.9, eps=1e-8, weight_decay=0.0):
        super().__init__(lr, weight_decay)
        self.rho, self.eps = rho, eps

    def _update(self, key, p, g):
        s = self.state.get(key)
        if s is None:
            s = np.zeros_like(p)
        s = self.rho * s + (1 - self.rho) * g * g
        self.state[key] = s
        p -= self.lr * g / (np.sqrt(s) + self.eps)


class Adam(Optimizer):
    def __init__(self, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.0):
        super().__init__(lr, weight_decay)
        self.b1, self.b2, self.eps = beta1, beta2, eps

    def _update(self, key, p, g):
        m, v = self.state.get(key, (np.zeros_like(p), np.zeros_like(p)))
        m = self.b1 * m + (1 - self.b1) * g
        v = self.b2 * v + (1 - self.b2) * g * g
        self.state[key] = (m, v)
        mh = m / (1 - self.b1 ** self.t)
        vh = v / (1 - self.b2 ** self.t)
        p -= self.lr * mh / (np.sqrt(vh) + self.eps)


def get_optimizer(name, lr, weight_decay=0.0, momentum=0.9):
    name = name.lower()
    if name == "sgd":
        return SGD(lr, momentum=0.0, weight_decay=weight_decay)
    if name == "momentum":
        return SGD(lr, momentum=momentum, weight_decay=weight_decay)
    if name == "nesterov":
        return SGD(lr, momentum=momentum, nesterov=True, weight_decay=weight_decay)
    if name == "rmsprop":
        return RMSprop(lr, weight_decay=weight_decay)
    if name == "adam":
        return Adam(lr, weight_decay=weight_decay)
    raise ValueError(f"Optimiseur inconnu : {name}")


## 3. Modèle LeNet-5
C1 (6@5×5) → S2 → C3 (16@5×5) → S4 → C5 (120) → F6 (84) → sortie (33).

In [ ]:


class Sequential:
    def __init__(self, layers):
        self.layers = layers

    def forward(self, x, training=True, return_acts=False):
        acts = {}
        for layer in self.layers:
            layer.training = training
            x = layer.forward(x)
            acts[layer.name] = x
        return (x, acts) if return_acts else x

    def backward(self, dout):
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout

    def n_params(self):
        return sum(p.size for l in self.layers if l.trainable for p in l.params.values())

    def get_state(self):
        return {f"{l.name}.{k}": v.copy() for l in self.layers if l.trainable
                for k, v in l.params.items()}

    def set_state(self, state):
        for l in self.layers:
            if l.trainable:
                for k in l.params:
                    l.params[k][...] = state[f"{l.name}.{k}"]

    def save(self, path):
        np.savez(path, **self.get_state())

    def load(self, path):
        self.set_state(dict(np.load(path)))

    def summary(self, input_shape=(1, 1, 32, 32)):
        x = np.zeros(input_shape, dtype=next(l for l in self.layers if l.trainable)
                     .params["W"].dtype)
        lines = [f"{'Couche':<8}{'Sortie':<20}{'Paramètres':>12}", "-" * 40]
        for l in self.layers:
            l.training = False
            x = l.forward(x)
            n = sum(p.size for p in l.params.values())
            lines.append(f"{l.name:<8}{str(x.shape[1:]):<20}{n:>12}")
        lines.append("-" * 40)
        lines.append(f"{'Total':<28}{self.n_params():>12}")
        return "\n".join(lines)


def build_lenet5(num_classes=33, in_ch=1, activation="tanh", pooling="avg",
                 c5_as_conv=False, dropout=0.0, init="lecun", seed=0,
                 dtype=np.float32):
    """LeNet-5 : C1-S2-C3-S4-C5-F6-OUT.

    C5 peut être implémentée comme une couche Dense (400->120) ou comme une
    convolution 5x5 sur 16@5x5 (120@1x1) ; les deux sont équivalentes.
    """
    rng = np.random.default_rng(seed)
    Pool = AvgPool2D if pooling == "avg" else MaxPool2D
    kw = dict(init=init, rng=rng, dtype=dtype)
    L = [Conv2D(in_ch, 6, 5, name="C1", **kw), get_activation(activation, "a1"),
         Pool(2, name="S2"),
         Conv2D(6, 16, 5, name="C3", **kw), get_activation(activation, "a3"),
         Pool(2, name="S4")]
    if c5_as_conv:
        L += [Conv2D(16, 120, 5, name="C5", **kw), get_activation(activation, "a5"),
              Flatten(name="flat")]
    else:
        L += [Flatten(name="flat"), Dense(16 * 5 * 5, 120, name="C5", **kw),
              get_activation(activation, "a5")]
    if dropout > 0:
        L.append(Dropout(dropout, rng=rng, name="drop"))
    L += [Dense(120, 84, name="F6", **kw), get_activation(activation, "a6"),
          Dense(84, num_classes, name="OUT", **kw)]
    return Sequential(L)


In [ ]:
model = build_lenet5(33)
print(model.summary())

## 4. Vérification des gradients (différences finies)
Chaque erreur relative doit être très petite (< 1e-5).

In [ ]:


def check(activation="tanh", pooling="avg", c5_as_conv=False, n_checks=6, eps=1e-5, seed=0):
    rng = np.random.default_rng(seed)
    model = build_lenet5(33, activation=activation, pooling=pooling,
                         c5_as_conv=c5_as_conv, seed=seed, dtype=np.float64)
    loss_fn = SoftmaxCrossEntropy()
    x = rng.normal(size=(4, 1, 32, 32))
    y = rng.integers(0, 33, size=4)

    loss_fn.forward(model.forward(x, training=False), y)
    model.backward(loss_fn.backward())

    def loss():
        return loss_fn.forward(model.forward(x, training=False), y)

    worst = 0.0
    for layer in model.layers:
        if not layer.trainable:
            continue
        for k, p in layer.params.items():
            g = layer.grads[k].copy()
            for _ in range(n_checks):
                idx = tuple(rng.integers(0, s) for s in p.shape)
                old = p[idx]
                p[idx] = old + eps; lp = loss()
                p[idx] = old - eps; lm = loss()
                p[idx] = old
                num = (lp - lm) / (2 * eps)
                rel = abs(num - g[idx]) / max(1e-8, abs(num) + abs(g[idx]))
                worst = max(worst, rel)
    return worst



for act, pool, c5 in [('tanh','avg',False), ('tanh','avg',True), ('relu','max',False), ('sigmoid','avg',False)]:
    w = check(act, pool, c5)
    print(f'{act:8s} pool={pool:3s} C5conv={c5!s:5s} erreur relative max = {w:.2e}', 'OK' if w < 1e-5 else 'ECHEC')

## 5. Données

In [ ]:

IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".gif")


def load_from_directory(root, size=32):
    """Dossier organisé en sous-dossiers par classe : root/<classe>/*.png

    Retourne X (N,1,size,size) float32 dans [0,1] (trait clair sur fond sombre),
    y (N,) int64 et la liste des noms de classes.
    """
    from PIL import Image
    classes = sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))
    if not classes:
        raise FileNotFoundError(f"Aucun sous-dossier de classe trouvé dans {root}")
    X, y = [], []
    for ci, c in enumerate(classes):
        folder = os.path.join(root, c)
        for f in sorted(os.listdir(folder)):
            if f.lower().endswith(IMG_EXT):
                img = Image.open(os.path.join(folder, f)).convert("L").resize((size, size))
                X.append(np.asarray(img, dtype=np.float32) / 255.0)
                y.append(ci)
    X = np.stack(X)[:, None]
    if X.mean() > 0.5:          # fond blanc -> on inverse pour avoir trait = valeur haute
        X = 1.0 - X
    return X, np.array(y, dtype=np.int64), classes


def load_from_npz(path):
    d = np.load(path, allow_pickle=True)
    X = d["X"].astype(np.float32)
    if X.ndim == 3:
        X = X[:, None]
    if X.max() > 1.5:
        X /= 255.0
    names = list(d["classes"]) if "classes" in d else [str(i) for i in range(int(d["y"].max()) + 1)]
    return X, d["y"].astype(np.int64), names


def stratified_split(y, val=0.15, test=0.15, seed=0):
    rng = np.random.default_rng(seed)
    tr, va, te = [], [], []
    for c in np.unique(y):
        idx = rng.permutation(np.where(y == c)[0])
        nt, nv = int(round(len(idx) * test)), int(round(len(idx) * val))
        te += list(idx[:nt]); va += list(idx[nt:nt + nv]); tr += list(idx[nt + nv:])
    return np.array(tr), np.array(va), np.array(te)


def standardize(Xtr, *others):
    mu, sd = Xtr.mean(), Xtr.std() + 1e-8
    return [(X - mu) / sd for X in (Xtr, *others)]


def augment_batch(xb, rng, max_shift=2):
    """Translation aléatoire (±max_shift px) — amélioration optionnelle."""
    out = np.empty_like(xb)
    pad = np.pad(xb, ((0, 0), (0, 0), (max_shift,) * 2, (max_shift,) * 2), mode="edge")
    H, W = xb.shape[2:]
    for i in range(len(xb)):
        dy, dx = rng.integers(0, 2 * max_shift + 1, size=2)
        out[i] = pad[i, :, dy:dy + H, dx:dx + W]
    return out


# --------------------------------------------------------------------------
# Jeu de données synthétique (uniquement pour tester le pipeline)
# --------------------------------------------------------------------------
def _draw_segment(img, p0, p1):
    t = np.linspace(0, 1, 60)
    ys = np.clip(np.round(p0[0] + t * (p1[0] - p0[0])).astype(int), 1, 30)
    xs = np.clip(np.round(p0[1] + t * (p1[1] - p0[1])).astype(int), 1, 30)
    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            img[ys + dy, xs + dx] = 1.0


def make_synthetic(n_per_class=80, num_classes=33, size=32, seed=0):
    """Glyphes aléatoires par classe (3-4 segments) + décalage, bruit, intensité."""
    rng = np.random.default_rng(seed)
    protos = []
    for _ in range(num_classes):
        img = np.zeros((size, size), dtype=np.float32)
        pts = rng.integers(6, 26, size=(rng.integers(4, 6), 2))
        for a, b in zip(pts[:-1], pts[1:]):
            _draw_segment(img, a, b)
        protos.append(img)
    X, y = [], []
    for c, p in enumerate(protos):
        for _ in range(n_per_class):
            dy, dx = rng.integers(-2, 3, size=2)
            im = np.roll(p, (dy, dx), axis=(0, 1)) * rng.uniform(0.7, 1.0)
            im = np.clip(im + rng.normal(0, 0.12, im.shape), 0, 1)
            X.append(im.astype(np.float32)); y.append(c)
    return np.stack(X)[:, None], np.array(y, dtype=np.int64), [f"classe_{i:02d}" for i in range(num_classes)]


### Choix de la source de données
Modifier `SOURCE` et les chemins ci-dessous. Pour la base AMHCD, le dossier d'images s'appelle `tifinagh-images` (33 sous-dossiers).

In [ ]:
SOURCE   = "zip"                     # "zip" | "dossier" | "synthetique"
ZIP_PATH = "amhcd-data-64.zip"       # chemin du .zip (si SOURCE == "zip")
DATA_DIR = "amhcd-data-64/tifinagh-images"   # dossier d'images (si SOURCE == "dossier")
SEED = 0

if SOURCE == "zip":
    import zipfile
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("amhcd_raw")
    # on cherche le dossier qui contient les 33 sous-dossiers de classes
    for root, dirs, files in os.walk("amhcd_raw"):
        if "__MACOSX" in root:
            continue
        if len(dirs) == 33 and os.path.basename(root) == "tifinagh-images":
            DATA_DIR = root
            break
    SOURCE = "dossier"

if SOURCE == "dossier":
    X, y, names = load_from_directory(DATA_DIR)
else:
    X, y, names = make_synthetic(80, seed=SEED)
    print("ATTENTION : données synthétiques (test du notebook uniquement)")

tr, va, te = stratified_split(y, seed=SEED)
Xtr, Xva, Xte = standardize(X[tr], X[va], X[te])
ytr, yva, yte = y[tr], y[va], y[te]
print("images :", X.shape, "| classes :", len(names))
print("train", Xtr.shape, "| val", Xva.shape, "| test", Xte.shape)
print("images par classe (min/max) :", np.bincount(y).min(), np.bincount(y).max())

fig, ax = plt.subplots(3, 11, figsize=(14, 4.5))
for c, a in enumerate(ax.ravel()):
    a.imshow(X[np.where(y == c)[0][0], 0], cmap="gray"); a.set_title(names[c], fontsize=8); a.axis("off")
plt.suptitle("Un exemple par classe"); plt.tight_layout(); plt.show()

## 6. Visualisations
Courbes, matrice de confusion, rapport par classe, cartes de caractéristiques, filtres.

In [ ]:


def plot_curves(history, path, title="LeNet-5 — apprentissage"):
    ep = np.arange(1, len(history["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(ep, history["train_loss"], label="train")
    ax[0].plot(ep, history["val_loss"], label="validation")
    ax[0].set(xlabel="Époque", ylabel="Loss (entropie croisée)", title="Loss")
    ax[1].plot(ep, history["train_acc"], label="train")
    ax[1].plot(ep, history["val_acc"], label="validation")
    ax[1].set(xlabel="Époque", ylabel="Accuracy", title="Accuracy")
    for a in ax:
        a.grid(alpha=.3); a.legend()
    fig.suptitle(title); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def confusion_matrix(y_true, y_pred, n):
    cm = np.zeros((n, n), dtype=np.int64)
    np.add.at(cm, (y_true, y_pred), 1)
    return cm


def plot_confusion_matrix(cm, class_names, path, normalize=True):
    n = len(cm)
    M = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1) if normalize else cm
    fig, ax = plt.subplots(figsize=(11, 9.5))
    im = ax.imshow(M, cmap="Blues", vmin=0)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(class_names, rotation=90, fontsize=7)
    ax.set_yticklabels(class_names, fontsize=7)
    ax.set(xlabel="Classe prédite", ylabel="Classe réelle",
           title="Matrice de confusion" + (" (normalisée par ligne)" if normalize else ""))
    fig.colorbar(im, fraction=0.046)
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def classification_report(cm, class_names, path=None):
    tp = np.diag(cm).astype(float)
    prec = tp / np.maximum(cm.sum(axis=0), 1)
    rec = tp / np.maximum(cm.sum(axis=1), 1)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    rows = [dict(classe=c, precision=round(p, 4), rappel=round(r, 4), f1=round(f, 4),
                 support=int(s)) for c, p, r, f, s in zip(class_names, prec, rec, f1, cm.sum(axis=1))]
    if path:
        with open(path, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
    return rows, dict(macro_f1=float(f1.mean()), accuracy=float(tp.sum() / cm.sum()))


def plot_feature_maps(model, x, path, layers=("C1", "S2", "C3", "S4"), max_maps=16):
    """x : une image (1,1,32,32) normalisée. Affiche les cartes de chaque couche."""
    _, acts = model.forward(x, training=False, return_acts=True)
    fig = plt.figure(figsize=(1.4 * max_maps / 2, 1.5 * (len(layers) + 1)))
    gs = fig.add_gridspec(len(layers) + 1, max_maps)
    ax = fig.add_subplot(gs[0, :2]); ax.imshow(x[0, 0], cmap="gray"); ax.set_title("Entrée", fontsize=8); ax.axis("off")
    for r, name in enumerate(layers, start=1):
        fm = acts[name][0]
        for j in range(min(len(fm), max_maps)):
            a = fig.add_subplot(gs[r, j]); a.imshow(fm[j], cmap="viridis"); a.axis("off")
            if j == 0:
                a.set_title(f"{name} {fm.shape[0]}@{fm.shape[1]}x{fm.shape[2]}", fontsize=8, loc="left")
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def plot_filters(model, path, layer="C1"):
    W = next(l for l in model.layers if l.name == layer).params["W"]
    n = W.shape[0]
    fig, axes = plt.subplots(1, n, figsize=(1.6 * n, 1.9))
    for i, a in enumerate(np.atleast_1d(axes)):
        a.imshow(W[i, 0], cmap="gray"); a.axis("off"); a.set_title(f"F{i + 1}", fontsize=8)
    fig.suptitle(f"Filtres appris — {layer}", fontsize=10)
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def plot_optimizer_comparison(histories, path):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for name, h in histories.items():
        ep = np.arange(1, len(h["val_loss"]) + 1)
        ax[0].plot(ep, h["val_loss"], label=name)
        ax[1].plot(ep, h["val_acc"], label=name)
    ax[0].set(xlabel="Époque", ylabel="Loss validation", title="Loss (validation)")
    ax[1].set(xlabel="Époque", ylabel="Accuracy validation", title="Accuracy (validation)")
    for a in ax:
        a.grid(alpha=.3); a.legend()
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


## 7. Entraînement et évaluation

In [ ]:
def iterate_minibatches(X, y, bs, rng, shuffle=True):
    idx = rng.permutation(len(X)) if shuffle else np.arange(len(X))
    for i in range(0, len(X), bs):
        b = idx[i:i + bs]
        yield X[b], y[b]


def evaluate(model, X, y, bs=256):
    loss_fn, total, preds = SoftmaxCrossEntropy(), 0.0, []
    for i in range(0, len(X), bs):
        logits = model.forward(X[i:i + bs], training=False)
        total += loss_fn.forward(logits, y[i:i + bs]) * len(logits)
        preds.append(logits.argmax(axis=1))
    preds = np.concatenate(preds)
    return total / len(X), float((preds == y).mean()), preds


def fit(model, opt, Xtr, ytr, Xva, yva, epochs=30, bs=64, seed=0, augment=False,
        lr_decay=1.0, patience=0, verbose=True):
    rng = np.random.default_rng(seed)
    loss_fn = SoftmaxCrossEntropy()
    hist = {k: [] for k in ("train_loss", "train_acc", "val_loss", "val_acc", "lr", "time")}
    best, best_state, bad = -1.0, None, 0
    for ep in range(1, epochs + 1):
        t0, tl, tc, n = time.time(), 0.0, 0, 0
        for xb, yb in iterate_minibatches(Xtr, ytr, bs, rng):
            if augment:
                xb = augment_batch(xb, rng)
            logits = model.forward(xb, training=True)
            tl += loss_fn.forward(logits, yb) * len(yb)
            tc += int((logits.argmax(1) == yb).sum()); n += len(yb)
            model.backward(loss_fn.backward())
            opt.step(model.layers)
        vl, va, _ = evaluate(model, Xva, yva)
        for k, v in zip(hist, (tl / n, tc / n, vl, va, opt.lr, time.time() - t0)):
            hist[k].append(float(v))
        if verbose:
            print(f"Ep {ep:3d}/{epochs} | loss {tl/n:.4f} acc {tc/n:.4f} | "
                  f"val_loss {vl:.4f} val_acc {va:.4f} | lr {opt.lr:.2e} | {hist['time'][-1]:.1f}s",
                  flush=True)
        if va > best:
            best, best_state, bad = va, model.get_state(), 0
        else:
            bad += 1
            if patience and bad >= patience:
                print("Arrêt anticipé."); break
        opt.lr *= lr_decay
    model.set_state(best_state)          # on restaure les meilleurs poids (validation)
    return hist


### 7.1 Modèle de base (Adam)
Paramètres modifiables : `EPOCHS`, `LR`, `OPTIMIZER` (`sgd`, `momentum`, `nesterov`, `rmsprop`, `adam`).

In [ ]:
OUT_DIR   = "runs/baseline"
OPTIMIZER = "adam"
LR        = 1e-3
EPOCHS    = 30
os.makedirs(OUT_DIR, exist_ok=True)

n_cls = len(names)
model = build_lenet5(n_cls, seed=SEED)
opt = get_optimizer(OPTIMIZER, LR)
hist = fit(model, opt, Xtr, ytr, Xva, yva, epochs=EPOCHS, bs=64, seed=SEED, lr_decay=0.95, patience=8)

tl, ta, pred = evaluate(model, Xte, yte)
print(f"\nTEST : loss {tl:.4f} | accuracy {ta:.4f} | erreurs {(pred != yte).sum()} / {len(yte)}")

In [ ]:
out = lambda f: os.path.join(OUT_DIR, f)
cm = confusion_matrix(yte, pred, n_cls)
rows, extra = classification_report(cm, names, out("rapport_classes.csv"))
print("F1 macro :", round(extra["macro_f1"], 4))

plot_curves(hist, out("courbes.png"));                          display(Image(out("courbes.png")))
plot_confusion_matrix(cm, names, out("matrice_confusion.png")); display(Image(out("matrice_confusion.png")))
plot_feature_maps(model, Xte[:1], out("cartes_caracteristiques.png")); display(Image(out("cartes_caracteristiques.png")))
plot_filters(model, out("filtres_C1.png"));                     display(Image(out("filtres_C1.png")))
model.save(out("poids.npz"))

# erreurs les plus fréquentes
cm_err = cm.copy(); np.fill_diagonal(cm_err, 0)
print("Confusions les plus fréquentes :")
for i, j in sorted(zip(*np.nonzero(cm_err)), key=lambda t: -cm_err[t])[:6]:
    print(f"  {names[i]} -> {names[j]} : {cm_err[i, j]}")

### 7.2 Comparaison des optimiseurs
20 époques par optimiseur (≈ 3 à 4 min chacun sur CPU avec la vraie base). Réduire `EPOCHS_CMP` pour aller plus vite.

In [ ]:
EPOCHS_CMP = 20
configs = {"SGD": ("sgd", 0.05), "Momentum": ("momentum", 0.01),
           "RMSprop": ("rmsprop", 1e-3), "Adam": ("adam", 1e-3)}
hs, summary = {}, {}
for label, (opt_name, lr) in configs.items():
    print(f"\n=== {label} (lr={lr}) ===")
    m = build_lenet5(n_cls, seed=SEED)
    h = fit(m, get_optimizer(opt_name, lr), Xtr, ytr, Xva, yva, EPOCHS_CMP, 64, SEED, verbose=False)
    hs[label] = h
    summary[label] = dict(lr=lr, best_val_acc=max(h["val_acc"]), test_acc=evaluate(m, Xte, yte)[1])
    print(f"  meilleure acc. validation {summary[label]['best_val_acc']:.4f} | test {summary[label]['test_acc']:.4f}")

os.makedirs("runs/compare", exist_ok=True)
plot_optimizer_comparison(hs, "runs/compare/comparaison_optimiseurs.png")
display(Image("runs/compare/comparaison_optimiseurs.png"))
print(json.dumps(summary, indent=2))

### 7.3 Variante améliorée (bonus)
ReLU + max-pooling + dropout + régularisation L2 (+ augmentation par translation si `AUGMENT = True`).

In [ ]:
AUGMENT = False
model2 = build_lenet5(n_cls, activation="relu", pooling="max", dropout=0.2, seed=SEED)
opt2 = get_optimizer("adam", 1e-3, weight_decay=1e-4)
hist2 = fit(model2, opt2, Xtr, ytr, Xva, yva, epochs=25, bs=64, seed=SEED, augment=AUGMENT, lr_decay=0.93, patience=8)
tl2, ta2, _ = evaluate(model2, Xte, yte)
print(f"\nTEST variante : accuracy {ta2:.4f} (modèle de base : {ta:.4f})")

## 8. Résultats de référence (obtenus avec la base AMHCD, 70/15/15, graine 0)
| Configuration | Accuracy test |
|---|---|
| LeNet-5 de base, Adam, 30 époques | **99,20 %** |
| SGD / Momentum / RMSprop / Adam (20 époques, lr constant) | 97,87 / 98,72 / 98,72 / 98,84 % |
| ReLU + max-pool + dropout 0,2 + L2 | 98,46 % |
| ReLU + max-pool + dropout 0,3 + L2 + translations | 97,30 % |

Une seule graine : les écarts inférieurs à ~0,5 point ne sont pas significatifs. Les résultats de votre exécution peuvent différer légèrement.